# Pipeline

> Run OCR + fix the markdown headings + describe images/figures for a single pdf file

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs, ocr_pdf
from mistocr.refine import add_img_descs, fix_hdgs
from pathlib import Path
from asyncio import Semaphore, gather, sleep
import os, json, shutil

In [ ]:
@delegates(add_img_descs)
async def pdf_to_md(
    pdf_path:str, # Path to input PDF file
    dst:str, # Destination directory for output markdown
    ocr_output:str=None, # Optional OCR output directory (defaults to pdf_path stem)
    model:str='claude-sonnet-4-5', # Model to use for heading fixes and image descriptions
    add_img_desc:bool=True, # Whether to add image descriptions
    progress:bool=True, # Whether to show progress messages
    **kwargs):
    "Convert PDF to markdown with OCR, fixed heading hierarchy, and optional image descriptions"
    n_steps = 3 if add_img_desc else 2
    if progress: print(f"Step 1/{n_steps}: Running OCR on {pdf_path}...")
    ocr_dirs = ocr_pdf(pdf_path, ocr_output or 'ocr_temp')
    ocr_dir = ocr_dirs[0]
    if progress: print(f"Step 2/{n_steps}: Fixing heading hierarchy...")
    fix_hdgs(ocr_dir, model=model)
    if add_img_desc:
        if progress: print(f"Step 3/{n_steps}: Adding image descriptions...")
        await add_img_descs(ocr_dir, dst=dst, model=model, progress=progress, **kwargs)
    elif dst and Path(dst) != ocr_dir: shutil.copytree(ocr_dir, dst, dirs_exist_ok=True)
    if progress: print("Done!")

In [ ]:
#| eval: false
await pdf_to_md('files/test/attention-is-all-you-need.pdf', 'files/test/md_test')

Step 1/3: Running OCR on files/test/attention-is-all-you-need.pdf...


Mistral batch job status: QUEUED


Mistral batch job status: RUNNING


Step 2/3: Fixing heading hierarchy...


Step 3/3: Adding image descriptions...
Loading existing descriptions from ocr_temp/attention-is-all-you-need/img_descriptions.json
Adding descriptions to 15 pages...
Done! Enriched pages saved to files/test/md_test
Done!


In [ ]:
#| eval: false
!ls -R files/test/md_test

files/test/md_test:
img	    page_11.md	page_14.md  page_3.md  page_6.md  page_9.md
page_1.md   page_12.md	page_15.md  page_4.md  page_7.md
page_10.md  page_13.md	page_2.md   page_5.md  page_8.md

files/test/md_test/img:
img-0.jpeg  img-1.jpeg	img-2.jpeg  img-3.jpeg	img-4.jpeg


In [ ]:
md = read_pgs('files/test/md_test', join=True)
print(md[5000:8000])


tions. In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more difficult to learn dependencies between distant positions [12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].
End-to-end memory networks are based on a recurrent attenti